In [ ]:
#Uploaded files in personal google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install mne loguru typer tqdm seaborn tensorflow scikit-learn

In [ ]:
Location of files
#%cd /content/drive/...

#Building Epochs
### Here we build epochs for boch Onset-locked and Offset-locked with metadata
### then it is simply usable for every scenario just like a query.

In [ ]:
from pathlib import Path
import gc
import re
import mne
import numpy as np
import pandas as pd

mne.use_log_level("warning")

#Location of datasets from EEG (https://zenodo.org/records/15332758?preview_file=undefined)
INPUT_DIR  = Path("/content/drive/...")

#Direction for onset and offset locked files
ONSET_DIR  = Path("/content/drive/...")
OFFSET_DIR = Path("/content/drive/...")
ONSET_DIR.mkdir(parents=True, exist_ok=True)
OFFSET_DIR.mkdir(parents=True, exist_ok=True)

ONSET_OPT_DICT = {50: 'Raw', 51: 'AM', 52: 'FM', 56: 'Binaural'}
ONSET_TGT_DICT = {70: 'Raw', 71: 'AM', 72: 'FM', 76: 'Binaural'}


def get_trigger(desc):
    m = re.search(r"Stimulus/S\s*(\d+)", desc)
    return int(m.group(1)) if m else None


def build_metadata(events):
    in_trial = False
    trial_events = []
    on_evts_list, on_meta_list = [], []
    off_evts_list, off_meta_list = [], []

    for ev in events:
        code = ev[2]
        if code == 101:
            in_trial = True
            trial_events = []
        elif code == 102 and in_trial:
            onsets = [e for e in trial_events if e[2] in ONSET_OPT_DICT or e[2] in ONSET_TGT_DICT]
            offsets = [e for e in trial_events if e[2] in {69, 89}]

            is_att_check = not any(e[2] in ONSET_TGT_DICT for e in onsets)
            trial_type = 'Attention_Check' if is_att_check else 'Normal'

            target_pos = -1
            if not is_att_check:
                for idx, e in enumerate(onsets):
                    if e[2] in ONSET_TGT_DICT:
                        target_pos = idx + 1
                        break

            for idx, e in enumerate(onsets):
                pos = idx + 1
                is_tgt = e[2] in ONSET_TGT_DICT
                tagger = ONSET_TGT_DICT[e[2]] if is_tgt else ONSET_OPT_DICT[e[2]]

                on_evts_list.append([e[0], 0, 2 if is_tgt else 1])
                on_meta_list.append({
                    'Trial_Type': trial_type,
                    'Word_Position': pos,
                    'Is_Target': is_tgt,
                    'Target_Position': target_pos,
                    'Tagger_Type': tagger,
                    'Original_Code': e[2]
                })

            for idx, e in enumerate(offsets):
                pos = idx + 1
                is_tgt = (e[2] == 89)
                tagger = 'Unknown'
                if idx < len(onsets):
                    onset_code = onsets[idx][2]
                    tagger = ONSET_TGT_DICT.get(onset_code, ONSET_OPT_DICT.get(onset_code, 'Unknown'))

                off_evts_list.append([e[0], 0, 2 if is_tgt else 1])
                off_meta_list.append({
                    'Trial_Type': trial_type,
                    'Word_Position': pos,
                    'Is_Target': is_tgt,
                    'Target_Position': target_pos,
                    'Tagger_Type': tagger,
                    'Original_Code': e[2]
                })

            in_trial = False
        elif in_trial:
            trial_events.append(ev)

    on_events_arr = np.array(on_evts_list) if on_evts_list else np.array([])
    on_meta_df = pd.DataFrame(on_meta_list) if on_meta_list else pd.DataFrame()
    off_events_arr = np.array(off_evts_list) if off_evts_list else np.array([])
    off_meta_df = pd.DataFrame(off_meta_list) if off_meta_list else pd.DataFrame()

    return on_events_arr, on_meta_df, off_events_arr, off_meta_df


def load_raw(raw_path, subj):
    raw = mne.io.read_raw_fif(raw_path, preload=False, verbose=False)
    #Subject 06 needs Merging, because of interrupt in experiment.
    if subj == "sub-06":
        part2_path = raw_path.parent / "sub-06_ses-01_2-raw.fif"
        if part2_path.exists():
            print("  -> Merging sub-06 part 1 and part 2...")
            raw2 = mne.io.read_raw_fif(part2_path, preload=False, verbose=False)
            events1, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
            events2, _ = mne.events_from_annotations(raw2, event_id=get_trigger, verbose=False)

            trigs_102 = events1[events1[:, 2] == 102]
            if len(trigs_102) > 0:
                raw.crop(tmin=0, tmax=trigs_102[-1, 0] / raw.info['sfreq'])

            trigs_start = events2[(events2[:, 2] == 100) | (events2[:, 2] == 101)]
            if len(trigs_start) > 0:
                raw2.crop(tmin=trigs_start[0, 0] / raw2.info['sfreq'])

            raw = mne.concatenate_raws([raw, raw2])
            del raw2
            gc.collect()

    raw.load_data()
    return raw


def process_single_subject(raw_path):
    subj = re.search(r"(sub-\d+)", raw_path.name).group(1)
    print(f"\n========================================")
    print(f"Processing {subj}...")
    print(f"========================================")

    # ===== ONSET=====
    raw = None
    epochs_onset = None
    try:
        raw = load_raw(raw_path, subj)
        events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
        on_events_arr, on_meta_df, _, _ = build_metadata(events)

        if len(on_events_arr) > 0:
            print("  -> Extracting ONSET epochs (no filter)...")
            epochs_onset = mne.Epochs(
                raw, on_events_arr,
                event_id={"option": 1, "target": 2},
                tmin=-0.2, tmax=1.0,
                baseline=(-0.2, 0.0),
                metadata=on_meta_df,
                picks="eeg",
                preload=True,
                reject_by_annotation=False,
                verbose=False,
            )
            out_path_on = ONSET_DIR / f"{subj}-onset-epo.fif"
            epochs_onset.save(out_path_on, overwrite=True)
            print(f"     Saved {out_path_on.name} — {len(epochs_onset)} epochs")

            del epochs_onset
            epochs_onset = None
            gc.collect()
    except Exception as e:
        print(f"  Error (onset) on {subj}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if raw is not None:
            del raw
        if epochs_onset is not None:
            del epochs_onset
        gc.collect()

    # ===== OFFSET: raw with high-pass 0.1Hz =====
    raw = None
    epochs_offset = None
    try:
        raw = load_raw(raw_path, subj)
        raw.filter(l_freq=0.1, h_freq=None, verbose=False)

        events, _ = mne.events_from_annotations(raw, event_id=get_trigger, verbose=False)
        _, _, off_events_arr, off_meta_df = build_metadata(events)

        if len(off_events_arr) > 0:
            print("  -> Extracting OFFSET epochs (high-pass 0.1Hz)...")
            epochs_offset = mne.Epochs(
                raw, off_events_arr,
                event_id={"option": 1, "target": 2},
                tmin=-0.5, tmax=0.6,
                baseline=None,
                metadata=off_meta_df,
                picks="eeg",
                preload=True,
                reject_by_annotation=False,
                verbose=False,
            )
            out_path_off = OFFSET_DIR / f"{subj}-offset-epo.fif"
            epochs_offset.save(out_path_off, overwrite=True)
            print(f"     Saved {out_path_off.name} — {len(epochs_offset)} epochs")

            del epochs_offset
            epochs_offset = None
            gc.collect()
    except Exception as e:
        print(f"  Error (offset) on {subj}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if raw is not None:
            del raw
        if epochs_offset is not None:
            del epochs_offset
        gc.collect()


raw_files = sorted(
    f for f in INPUT_DIR.glob("sub-*_ses-01-raw.fif")
    if "sub-03" not in f.name and "_2" not in f.name and "sub-michal" not in f.name
)
print(f"Found {len(raw_files)} subjects. Starting Unified Epoching with Metadata...")

for raw_path in raw_files:
    process_single_subject(raw_path)

print("\nALL UNIFIED EPOCHS DONE!")